# 04 — UCI Baseline Model

## Objective

This notebook builds and evaluates a baseline model following the main
workflow described on the UCI SECOM dataset page.

The workflow includes:

1. Handling missing values
2. Removing constant features
3. Standardizing measurement features
4. Selecting 40 features using the F-test
5. Training a Kernel Ridge model
6. Evaluating the model using 10-fold cross-validation

## Main Questions

1. Can the model identify Fail samples better than the Always-Pass baseline?
2. How well does the model identify both Fail and Pass samples?
3. What are the model's TPR, TNR, and Balanced Error Rate?
4. Are the results consistent across the 10 validation folds?
5. How do the results compare with the UCI reference results?

## 4.1 Data Preparation

The measurement data and Pass/Fail labels are loaded from the original
SECOM files.

The feature matrix contains the anonymized manufacturing measurements,
while the target vector contains the corresponding Pass/Fail labels.

Before building the model, this section verifies:

1. The number of samples and measurement features
2. The number of Pass and Fail samples
3. The number of missing measurement values
4. The number of constant features observed in the full dataset
5. The alignment between the feature and label row counts

These checks confirm that the data has been loaded correctly before
preprocessing and model training.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.kernel_ridge import KernelRidge

from sklearn.metrics import confusion_matrix

In [2]:
data_dir = Path("../data/raw")

feature_path = data_dir / "secom.data"
label_path = data_dir / "secom_labels.data"

X = pd.read_csv(
    feature_path,
    sep=r"\s+",
    header=None
)

X.columns = [
    f"feature_{i}"
    for i in range(X.shape[1])
]

labels = pd.read_csv(
    label_path,
    sep=r"\s+",
    header=None,
    names=["label", "timestamp_text"]
)

y = labels["label"]

In [3]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass counts:")
print(y.value_counts().sort_index())

print("\nMissing measurement values:", X.isna().sum().sum())
print("Constant features before preprocessing:",
      (X.nunique(dropna=True) == 1).sum())

assert len(X) == len(y)
assert set(y.unique()) == {-1, 1}

X shape: (1567, 590)
y shape: (1567,)

Class counts:
label
-1    1463
 1     104
Name: count, dtype: int64

Missing measurement values: 41951
Constant features before preprocessing: 116


In [4]:
from sklearn.pipeline import Pipeline

## 4.2 Preprocessing Pipeline

The SECOM data cannot be passed directly to the model because it contains
missing values, constant features, and measurement variables with
different numerical scales.

The preprocessing workflow is:

1. Fill missing values using the training-set median.
2. Remove features with zero variance.
3. Standardize the remaining features.
4. Select the 40 highest-ranked features using the F-test.

All preprocessing rules must be learned only from the training data
within each cross-validation fold to prevent data leakage.

In [5]:
preprocessor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "constant_removal",
        VarianceThreshold(threshold=0.0)
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "feature_selection",
        SelectKBest(
            score_func=f_classif,
            k=40
        )
    )
])

preprocessor

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('constant_removal', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimat

### Implementation Note

The UCI page confirms that the dataset contains missing values but does
not specify the imputation method used in the baseline experiment.

Median imputation is used here because it is less sensitive to extreme
values than mean imputation.

## 4.3 Single-Fold Preprocessing Check

Before running the complete 10-fold experiment, the preprocessing
pipeline is tested using the first cross-validation fold.

The objectives are to verify that:

1. The training and validation samples are separated correctly.
2. The preprocessing rules are learned only from the training fold.
3. The validation fold is transformed using the training rules.
4. No missing values remain after preprocessing.
5. The final feature matrix contains 40 selected features.

In [6]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

### Cross-validation Choice

The UCI baseline uses 10-fold cross-validation but does not specify
whether the folds were stratified or shuffled.

Stratified folds are used here because Fail samples are rare. This helps
each validation fold contain a similar number of Pass and Fail samples.
Shuffling and a fixed random seed are used to make the experiment
reproducible.

In [7]:
train_index, validation_index = next(cv.split(X, y))

X_train = X.iloc[train_index]
y_train = y.iloc[train_index]

X_validation = X.iloc[validation_index]
y_validation = y.iloc[validation_index]

In [8]:
print("Training feature shape:", X_train.shape)
print("Training label shape:", y_train.shape)

print("\nValidation feature shape:", X_validation.shape)
print("Validation label shape:", y_validation.shape)

print("\nTraining class counts:")
print(y_train.value_counts().sort_index())

print("\nValidation class counts:")
print(y_validation.value_counts().sort_index())

Training feature shape: (1410, 590)
Training label shape: (1410,)

Validation feature shape: (157, 590)
Validation label shape: (157,)

Training class counts:
label
-1    1316
 1      94
Name: count, dtype: int64

Validation class counts:
label
-1    147
 1     10
Name: count, dtype: int64


In [9]:
X_train_processed = preprocessor.fit_transform(
    X_train,
    y_train
)

X_validation_processed = preprocessor.transform(
    X_validation
)

In [10]:
print("Before preprocessing:")
print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)

print("\nAfter preprocessing:")
print("Training shape:", X_train_processed.shape)
print("Validation shape:", X_validation_processed.shape)

print("\nMissing values after preprocessing:")
print("Training:", np.isnan(X_train_processed).sum())
print("Validation:", np.isnan(X_validation_processed).sum())

Before preprocessing:
Training shape: (1410, 590)
Validation shape: (157, 590)

After preprocessing:
Training shape: (1410, 40)
Validation shape: (157, 40)

Missing values after preprocessing:
Training: 0
Validation: 0


## 4.4 Selected Features in the First Fold

The F-test ranks the remaining measurement features according to how
strongly their values differ between the Pass and Fail groups in the
first training fold.

The 40 highest-ranked features are retained for model training.

This section identifies the selected anonymous features and their
F-test scores. These scores indicate statistical association with the
label, but they do not establish physical causality.

In [11]:
constant_step = preprocessor.named_steps["constant_removal"]

non_constant_mask = constant_step.get_support()

non_constant_feature_names = X.columns[non_constant_mask]

print("Original number of features:", X.shape[1])
print("Features removed as constant:",
      (~non_constant_mask).sum())
print("Features remaining after constant removal:",
      non_constant_mask.sum())

Original number of features: 590
Features removed as constant: 116
Features remaining after constant removal: 474


In [12]:
selection_step = preprocessor.named_steps["feature_selection"]

selected_mask = selection_step.get_support()

selected_feature_names = non_constant_feature_names[selected_mask]

print("Number of selected features:",
      len(selected_feature_names))

print("\nSelected features:")
print(selected_feature_names.tolist())

Number of selected features: 40

Selected features:
['feature_21', 'feature_22', 'feature_26', 'feature_28', 'feature_59', 'feature_64', 'feature_103', 'feature_114', 'feature_121', 'feature_123', 'feature_124', 'feature_125', 'feature_129', 'feature_130', 'feature_133', 'feature_159', 'feature_160', 'feature_163', 'feature_164', 'feature_165', 'feature_180', 'feature_210', 'feature_247', 'feature_249', 'feature_294', 'feature_295', 'feature_298', 'feature_299', 'feature_300', 'feature_316', 'feature_348', 'feature_387', 'feature_430', 'feature_431', 'feature_434', 'feature_435', 'feature_436', 'feature_437', 'feature_452', 'feature_510']


In [13]:
feature_scores = pd.DataFrame({
    "feature": non_constant_feature_names,
    "f_score": selection_step.scores_
})

selected_feature_scores = (
    feature_scores[
        feature_scores["feature"].isin(
            selected_feature_names
        )
    ]
    .sort_values(
        by="f_score",
        ascending=False
    )
    .reset_index(drop=True)
)

selected_feature_scores

,feature,f_score
0,feature_59,30.098397
1,feature_431,26.251969
2,feature_103,26.233486
3,feature_348,25.948907
4,feature_430,21.919555
5,feature_21,21.700825
6,feature_434,21.563437
7,feature_435,20.575108
8,feature_436,19.806891
9,feature_510,19.392310


### Interpretation

The preprocessing pipeline retained 40 anonymous measurement features
for the first fold.

The selected features had the highest F-test scores within the first
training fold, indicating relatively strong statistical differences
between the Pass and Fail groups.

Because feature selection was fitted only on the first training fold,
the selected feature set may change in other cross-validation folds.
In addition, a high F-test score indicates association with the label,
not a causal relationship with manufacturing failure.

## 4.5 Kernel Ridge Model

A Kernel Ridge model is trained using the 40 features selected from the
first training fold.

The model produces a continuous score for each validation sample.
Because the original labels are `-1` for Pass and `+1` for Fail, a
threshold of zero is used to convert the scores into class predictions:

- Score below `0` → Pass (`-1`)
- Score equal to or above `0` → Fail (`+1`)

### Implementation Note

The UCI page does not report the complete Kernel Ridge parameter
settings used in its baseline experiment.

This implementation uses:

- a linear kernel;
- `alpha=1.0`;
- a decision threshold of `0.0`.

These settings provide a transparent starting point and should not be
interpreted as an exact reproduction of the original UCI model.

In [15]:
kernel_ridge_model = KernelRidge(
    alpha=1.0,
    kernel="linear"
)

kernel_ridge_model.fit(
    X_train_processed,
    y_train
)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`. If an array is passed, penalties areassumed to be specific to the targets. Hence they must correspond innumber. See :ref:`ridge_regression` for formula.",1.0
,"kernel kernel: str or callable, default=""linear""Kernel mapping used internally. This parameter is directly passed to:class:`~sklearn.metrics.pairwise.pairwise_kernels`.If `kernel` is a string, it must be one of the metricsin `pairwise.PAIRWISE_KERNEL_FUNCTIONS` or ""precomputed"".If `kernel` is ""precomputed"", X is assumed to be a kernel matrix.Alternatively, if `kernel` is a callable function, it is called oneach pair of instances (rows) and the resulting value recorded. Thecallable should take two rows from X as input and return thecorresponding kernel value as a single number. This means thatcallables from :mod:`sklearn.metrics.pairwise` are not allowed, asthey operate on matrices, not single samples. Use the stringidentifying the kernel instead.",'linear'
,"gamma gamma: float, default=NoneGamma parameter for the RBF, laplacian, polynomial, exponential chi2and sigmoid kernels. Interpretation of the default value is left tothe kernel; see the documentation for sklearn.metrics.pairwise.Ignored by other kernels.",None
,"degree degree: float, default=3Degree of the polynomial kernel. Ignored by other kernels.",3
,"coef0 coef0: float, default=1Zero coefficient for polynomial and sigmoid kernels.Ignored by other kernels.",1
,"kernel_params kernel_params: dict, default=NoneAdditional parameters (keyword arguments) for kernel function passedas callable object.",None
Name,Type,Value
"X_fit_ X_fit_: {ndarray, sparse matrix} of shape (n_samples, n_features)Training data, which is also required for prediction. Ifkernel == ""precomputed"" this is instead the precomputedtraining matrix, of shape (n_samples, n_samples).","ndarray[float64](1410, 40)","[[ 0.33, 0.74,-0.91,...,-0.55, 0.63, 0.23], [ 0.29,-0.33, 0.41,...,-0.49,-0.27, 2.22], [ 0.28, 0.01, 0.49,...,-0.54,-0.52, 4.8 ], ..., [ 0.76, 0.02, 0.02,..., 0.28, 0.4 ,-0.03], [ 0.57,-0.08, 0.26,...,-0.55, 1.99, 0.58], [ 0.49,-0.11, 0.37,..., 0.8 , 1.17, 0.52]]"
"dual_coef_ dual_coef_: ndarray of shape (n_samples,) or (n_samples, n_targets)Representation of weight vector(s) in kernel space","ndarray[float64](1410,)","[-1.07,-1.01, 0.72,...,-1.01,-0.85,-0.95]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,40


In [16]:
validation_scores = kernel_ridge_model.predict(
    X_validation_processed
)

print("Number of validation scores:",
      len(validation_scores))

print("Minimum score:",
      validation_scores.min())

print("Maximum score:",
      validation_scores.max())

print("First 10 scores:")
print(validation_scores[:10])

Number of validation scores: 157
Minimum score: -0.25063318756429
Maximum score: 1.388035729298224
First 10 scores:
[ 0.0621229   0.03901489  0.2548838   0.05605343  0.09505193  0.30659397
  0.37206783 -0.07114726  0.05156478  0.15162593]


In [17]:
decision_threshold = 0.0

y_validation_pred = np.where(
    validation_scores >= decision_threshold,
    1,
    -1
)

print("Predicted class counts:")
print(
    pd.Series(
        y_validation_pred,
        name="predicted_label"
    ).value_counts().sort_index()
)

Predicted class counts:
predicted_label
-1    99
 1    58
Name: count, dtype: int64


## 4.6 Single-Fold Evaluation

The predicted labels are compared with the true validation labels using
a confusion matrix.

Fail (`+1`) is treated as the positive class, while Pass (`-1`) is
treated as the negative class.

The evaluation reports:

- True Positive Rate: proportion of actual Fail samples detected
- True Negative Rate: proportion of actual Pass samples correctly identified
- Balanced Error Rate: average of the Fail error rate and Pass error rate

In [18]:
tn, fp, fn, tp = confusion_matrix(
    y_validation,
    y_validation_pred,
    labels=[-1, 1]
).ravel()

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

True Negatives: 98
False Positives: 49
False Negatives: 1
True Positives: 9


In [19]:
tpr = tp / (tp + fn)
tnr = tn / (tn + fp)

ber = (
    (1 - tpr) +
    (1 - tnr)
) / 2

accuracy = (
    tp + tn
) / (
    tp + tn + fp + fn
)

print("Accuracy:", round(accuracy, 4))
print("TPR:", round(tpr, 4))
print("TNR:", round(tnr, 4))
print("BER:", round(ber, 4))

Accuracy: 0.6815
TPR: 0.9
TNR: 0.6667
BER: 0.2167


## 4.7 Ten-Fold Cross-Validation

The complete preprocessing and modelling workflow is now evaluated
using all 10 cross-validation folds.

For each fold:

1. Nine folds are used as training data.
2. Missing-value imputation is learned from the training data.
3. Constant features are identified using the training data.
4. Standardization parameters are learned from the training data.
5. The 40 highest-ranked features are selected using the training data.
6. A new Kernel Ridge model is trained.
7. The remaining fold is used for validation.
8. TPR, TNR, BER, and accuracy are calculated.

A new preprocessing pipeline and model are fitted independently in
every fold to prevent data leakage.

In [20]:
baseline_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "constant_removal",
        VarianceThreshold(threshold=0.0)
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "feature_selection",
        SelectKBest(
            score_func=f_classif,
            k=40
        )
    ),
    (
        "model",
        KernelRidge(
            alpha=1.0,
            kernel="linear"
        )
    )
])

baseline_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('constant_removal', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimat

In [21]:
from sklearn.base import clone

In [22]:
fold_results = []
selected_features_by_fold = {}

for fold_number, (train_index, validation_index) in enumerate(
    cv.split(X, y),
    start=1
):
    # Split the current fold
    X_train_fold = X.iloc[train_index]
    y_train_fold = y.iloc[train_index]

    X_validation_fold = X.iloc[validation_index]
    y_validation_fold = y.iloc[validation_index]

    # Create and fit a fresh pipeline
    fold_model = clone(baseline_pipeline)

    fold_model.fit(
        X_train_fold,
        y_train_fold
    )

    # Generate continuous validation scores
    validation_scores_fold = fold_model.predict(
        X_validation_fold
    )

    # Convert scores into -1 / +1 predictions
    y_pred_fold = np.where(
        validation_scores_fold >= 0.0,
        1,
        -1
    )

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_validation_fold,
        y_pred_fold,
        labels=[-1, 1]
    ).ravel()

    # Evaluation metrics
    tpr_fold = tp / (tp + fn)
    tnr_fold = tn / (tn + fp)

    ber_fold = (
        (1 - tpr_fold) +
        (1 - tnr_fold)
    ) / 2

    accuracy_fold = (
        tp + tn
    ) / (
        tp + tn + fp + fn
    )

    # Save fold results
    fold_results.append({
        "fold": fold_number,
        "validation_samples": len(validation_index),
        "actual_fail": tp + fn,
        "predicted_fail": tp + fp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": accuracy_fold,
        "tpr": tpr_fold,
        "tnr": tnr_fold,
        "ber": ber_fold
    })

    # Recover selected feature names
    constant_mask = (
        fold_model
        .named_steps["constant_removal"]
        .get_support()
    )

    non_constant_names = X.columns[constant_mask]

    selection_mask = (
        fold_model
        .named_steps["feature_selection"]
        .get_support()
    )

    selected_features_by_fold[fold_number] = (
        non_constant_names[selection_mask].tolist()
    )

In [23]:
fold_results_df = pd.DataFrame(fold_results)

display_columns = [
    "fold",
    "actual_fail",
    "predicted_fail",
    "tn",
    "fp",
    "fn",
    "tp",
    "accuracy",
    "tpr",
    "tnr",
    "ber"
]

fold_results_df[display_columns].round(4)

,fold,actual_fail,predicted_fail,tn,fp,fn,tp,accuracy,tpr,tnr,ber
0,1,10,58,98,49,1,9,0.6815,0.9000,0.6667,0.2167
1,2,10,75,81,66,1,9,0.5732,0.9000,0.5510,0.2745
2,3,10,66,88,59,3,7,0.6051,0.7000,0.5986,0.3507
3,4,11,60,93,53,4,7,0.6369,0.6364,0.6370,0.3633
4,5,11,63,92,54,2,9,0.6433,0.8182,0.6301,0.2758
5,6,11,57,97,49,3,8,0.6688,0.7273,0.6644,0.3042
6,7,11,71,83,63,3,8,0.5796,0.7273,0.5685,0.3521
7,8,10,65,85,61,6,4,0.5705,0.4000,0.5822,0.5089
8,9,10,78,75,71,3,7,0.5256,0.7000,0.5137,0.3932
9,10,10,72,82,64,2,8,0.5769,0.8000,0.5616,0.3192


## 4.8 Cross-Validation Summary

The fold-level results are summarized using the mean and standard
deviation across the 10 validation folds.

The mean describes the overall cross-validation performance, while the
standard deviation indicates how sensitive the results are to different
training and validation samples.

In [25]:
summary_metrics = (
    fold_results_df[
        ["accuracy", "tpr", "tnr", "ber"]
    ]
    .agg(["mean", "std"])
    .T
)

summary_metrics

,mean,std
accuracy,0.606161,0.049809
tpr,0.730909,0.145278
tnr,0.597386,0.051020
ber,0.335852,0.079807


In [26]:
summary_percent = (
    summary_metrics * 100
).round(2)

summary_percent.columns = [
    "mean_percent",
    "std_percent"
]

summary_percent

,mean_percent,std_percent
accuracy,60.62,4.98
tpr,73.09,14.53
tnr,59.74,5.10
ber,33.59,7.98


## 4.8 Comparison with the UCI Reference

The cross-validation results are compared with the F-test baseline
reported on the UCI SECOM dataset page.

The comparison is indicative rather than an exact replication because
the UCI page does not provide the complete missing-value treatment,
Kernel Ridge parameters, fold construction, or decision threshold.

In [27]:
comparison = pd.DataFrame({
    "metric": ["TPR", "TNR", "BER"],
    "this_experiment_mean": [
        summary_percent.loc["tpr", "mean_percent"],
        summary_percent.loc["tnr", "mean_percent"],
        summary_percent.loc["ber", "mean_percent"]
    ],
    "this_experiment_std": [
        summary_percent.loc["tpr", "std_percent"],
        summary_percent.loc["tnr", "std_percent"],
        summary_percent.loc["ber", "std_percent"]
    ],
    "uci_f_test_mean": [
        59.1,
        73.8,
        33.5
    ],
    "uci_f_test_reported_variation": [
        4.8,
        1.8,
        2.2
    ]
})

comparison

,metric,this_experiment_mean,this_experiment_std,uci_f_test_mean,uci_f_test_reported_variation
0,TPR,73.09,14.53,59.1,4.8
1,TNR,59.74,5.10,73.8,1.8
2,BER,33.59,7.98,33.5,2.2


## 4.9 Overall Out-of-Fold Results

Each sample appears in a validation fold exactly once. Therefore, the
confusion-matrix counts from all 10 folds can be combined to summarize
the out-of-fold predictions across the complete dataset.

In [28]:
overall_tn = fold_results_df["tn"].sum()
overall_fp = fold_results_df["fp"].sum()
overall_fn = fold_results_df["fn"].sum()
overall_tp = fold_results_df["tp"].sum()

overall_results = pd.Series({
    "TN": overall_tn,
    "FP": overall_fp,
    "FN": overall_fn,
    "TP": overall_tp,
    "total_predictions": (
        overall_tn +
        overall_fp +
        overall_fn +
        overall_tp
    )
})

overall_results

TN                    874
FP                    589
FN                     28
TP                     76
total_predictions    1567
dtype: int64

In [29]:
assert overall_tn + overall_fp == 1463
assert overall_fn + overall_tp == 104
assert overall_results["total_predictions"] == 1567

In [30]:
overall_precision = overall_tp / (
    overall_tp + overall_fp
)

print(
    "Overall Fail precision:",
    round(overall_precision, 4)
)

Overall Fail precision: 0.1143


## 4.10 Baseline Summary

The leakage-safe baseline used median imputation, constant-feature
removal, feature standardization, F-test selection of 40 features, and
a linear Kernel Ridge model within stratified 10-fold cross-validation.

The model achieved:

- Mean TPR: 73.09%
- Mean TNR: 59.74%
- Mean BER: 33.59%
- Mean accuracy: 60.62%

Across the out-of-fold predictions, the model detected 76 of the 104
Fail samples but missed 28. It also incorrectly classified 589 of the
1,463 Pass samples as Fail.

The mean BER was close to the UCI F-test reference result of 33.5%.
However, the class-specific behavior was different: this implementation
achieved higher Fail detection but lower Pass recognition than the UCI
reference.

The relatively large variation in TPR shows that Fail detection was not
stable across folds. This is partly related to the small number of Fail
samples in each validation fold.

Overall, the baseline demonstrates that the measurement features contain
predictive information beyond the Always-Pass rule. However, the large
number of false alarms means that the current model is not yet suitable
as a practical production warning system.

These results should not be interpreted as evidence that the selected
anonymous features physically cause manufacturing failures.

The Kernel Ridge result is retained as a reproducible UCI reference
baseline. The next chapter evaluates probabilistic linear and nonlinear
models under the same cross-validation design to determine whether a more
useful risk-ranking and false-alarm trade-off can be achieved.